# lora-appraisal-judge — Kaggle launcher

This notebook only launches. All the logic (baselines, LoRA training, evaluation,
checkpointing, resume) lives in
[`scripts/train_and_evaluate.py`](https://github.com/carlosherreraUMA/lora-appraisal-judge/blob/master/scripts/train_and_evaluate.py)
in the repository. The next cell clones the repo, or resets it to the latest commit,
so a fix pushed there is picked up by re-running this notebook. No cell here should
ever need editing by hand.

**Settings (right-hand panel):** Accelerator → **GPU T4** (x2 is fine, the script
uses one) · Internet → **On**.

**Order:** run steps 0 and 1 interactively and read the smoke verdict. If it says OK,
run step 2. For a long unattended run, use **Save Version → Save & Run All
(Commit)**: it runs in the background with the browser closed, and whatever is in
`/kaggle/working` shows up in that version's **Output** tab.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/carlosherreraUMA/lora-appraisal-judge"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "origin/master"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

# The commit being run: compare it with the latest one on GitHub if in doubt.
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %cd %s", "--date=short"], check=True)

In [ ]:
# Installs what the image lacks and removes what breaks it. The steps live in the
# repo (scripts/kaggle_setup.sh), so an environment fix arrives by re-cloning.
!bash {REPO_DIR}/scripts/kaggle_setup.sh

## Step 1 — smoke test (about 5 minutes)

Trains 20 steps and scores 20 test examples, then prints the GPU, the dtype chosen
for it, the measured seconds per step, the projected time of the full run and a
**VERDICT**. Do not skip it: it is what would have caught the first attempt, which
ran eight hours at a tenth of the expected speed.

In [ ]:
!python -u {REPO_DIR}/scripts/train_and_evaluate.py --smoke --out /kaggle/working/smoke

## Step 2 — full run

Refuses to start unless the smoke report above found no problems. Checkpoints every
50 steps and writes each scored test example as it goes: if the session dies, run the
notebook again and it resumes from where it stopped.

In [ ]:
!python -u {REPO_DIR}/scripts/train_and_evaluate.py --out /kaggle/working/run --after-smoke /kaggle/working/smoke

## Results

`/kaggle/working/run/results.json`: the four metrics for majority, length and LoRA,
plus the environment (GPU, dtype, library versions) and training timings. Paste it
back to Claude to fill in `EXPERIMENTS.md`.

In [ ]:
print(open("/kaggle/working/run/results.json").read())